# 02. ImageNet-képek osztályozása DoReFaNet hálózattal

Az **QNN-MO-PYNQ** telepített modelljével egy címkézett ImageNet-mintát osztályozunk. Az első konvolúció és a teljesen összekapcsolt rétegek az **ARM processzoron** futnak; a köztes kvantált rétegek az **FPGA-n** hajtódnak végre, 1 bites súlyokkal és 2 bites aktivációkkal.

![DoReFaNet topológia](../dorefanet-topology.svg)

A cellákat sorrendben, **Shift + Enter** billentyűkkel futtassa. Előbb zárja be a többi FPGA-t használó Jupyter-kernelt. A teljesen összekapcsolt rétegek nagy súlyállományainak betöltéséhez több memória kellhet, mint amennyi a PYNQ-Z2 fedélzetén rendelkezésre álló 512 MB.

**Fontos:** a DoReFaNet PYNQ-Z2-n történő betöltése előtt hozzon létre és kapcsoljon be egy 1 GiB-os swapfájlt az alábbi útmutató szerint.

Induljon új kernellel. Ez a változat automatikusan megtalálja a telepített Python-csomagot, és a meglévő modellt, illetve képeket használja; nincs szükség újratelepítésre vagy modellletöltésre. A nem használt `Xlnk` importokat korábban el kell távolítani a `qnn/dorefanet.py` és `qnn/tinieryolo.py` fájlokból, a projekt útmutatója szerint.

*Az oktatási laboratórium fordítása; az inferenciakód az angol változatból származik. Ezt a fordított példányt külön, valódi hardveren még nem validáltuk.*

## 1. Könyvtárak importálása és fájlok megkeresése

Futtassa a következő cellát. Ez importálja a szükséges csomagokat, ellenőrzi a modell- és mintafájlokat, majd kiírja a használt könyvtárakat. **Az FPGA-t még nem inicializálja.**

A notebook rendszerint a telepített QNN notebookkönyvtárban fut. Ellenőrzi a szülőkönyvtárat is, majd szükség esetén a kártya szabványos útvonalát. Ha kell, állítsa be a `NOTEBOOK_DIR` változót arra a könyvtárra, amelyben az `imagenet-classes.pkl` és az `imagenet-samples` található.

In [ ]:
import os
import gc
import pickle
import random
import time
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt
from matplotlib.ticker import PercentFormatter
from PIL import Image
from IPython.display import display

import qnn
from qnn import Dorefanet, utils

QNN_ROOT = Path(qnn.__file__).resolve().parent
QNN_PARAM_DIR = QNN_ROOT / "params"
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "imagenet-classes.pkl").is_file():
    if (NOTEBOOK_DIR.parent / "imagenet-classes.pkl").is_file():
        NOTEBOOK_DIR = NOTEBOOK_DIR.parent
    else:
        NOTEBOOK_DIR = Path("/home/xilinx/jupyter_notebooks/qnn")
SAMPLE_DIR = NOTEBOOK_DIR / "imagenet-samples"
LABEL_FILE = NOTEBOOK_DIR / "imagenet-classes.pkl"

required_files = [QNN_PARAM_DIR / name for name in (
    "dorefanet-layers.json", "dorefanet-conv0.npy",
    "dorefanet-fc-normalized.npy", "ilsvrc_pp_mean.npy")]
required_files.append(LABEL_FILE)
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError("Required files missing:\n" + "\n".join(missing))
if not SAMPLE_DIR.is_dir():
    raise FileNotFoundError("Sample folder missing: " + str(SAMPLE_DIR))
image_files = sorted(path for path in SAMPLE_DIR.iterdir()
                     if path.is_file() and path.suffix.lower() in
                     (".jpg", ".jpeg", ".png", ".bmp"))
if not image_files:
    raise FileNotFoundError("No sample images found in " + str(SAMPLE_DIR))
print("Python package:", QNN_ROOT)
print("Notebook data:", NOTEBOOK_DIR)
print("Sample images:", len(image_files))

## 2. Kötelező memóriabeállítás a PYNQ-Z2-n

A PYNQ-Z2 mindössze **512 MB RAM-mal** rendelkezik. A DoReFaNet teljesen összekapcsolt rétegeinek súlyai betöltéskor kimeríthetik a memóriát és újraindíthatják a Jupyter-kernelt. Az első betöltés előtt ezért hozzon létre és aktiváljon egy **1 GiB-os swapfájlt**.

Nyisson terminált a PYNQ-Z2-n, és futtassa:

```bash
sudo dd if=/dev/zero of=/home/xilinx/qnn-install.swap bs=1M count=1024 status=progress
sudo chmod 600 /home/xilinx/qnn-install.swap
sudo mkswap /home/xilinx/qnn-install.swap
sudo swapon /home/xilinx/qnn-install.swap
```

Ellenőrizze, hogy a swap aktív-e:

```bash
free -h
swapon --show
```

A `free -h` kimenetében körülbelül **1 GiB swapnak** kell megjelennie.

### A kártya újraindítása után

A swapfájl megmarad az SD-kártyán, de újraindítás után nem aktiválódik automatikusan. **Ne hozza létre újra.** A meglévő fájlt ezzel aktiválja:

```bash
sudo swapon /home/xilinx/qnn-install.swap
```

Ezután ellenőrizze ismét:

```bash
free -h
```

### Ha a Jupyter-kernel leáll a súlyok betöltése közben

Jellemző hibaüzenet: **The kernel appears to have died. It will restart automatically.** Ez gyakran a következő sor után jelenik meg:

```text
Loading fully connected weights; please wait...
```

még az `Initializing the FPGA...` üzenet előtt. Zárja be a többi Jupyter-kernelt, ellenőrizze a `swapon --show` kimenetét, szükség esetén aktiválja a meglévő swapot, indítsa újra a notebook kernelét, majd futtasson mindent az elejétől.

## 3. Súlyok és osztálycímkék betöltése, a modell inicializálása

Futtassa a következő cellát, és várja meg a **Model ready** üzenetet. A program betölti a meglévő NumPy súlyszótárakat és az 1000 ImageNet-osztály címkéjét, majd inicializálja az FPGA-gyorsítót. A telepített csomag alapértelmezett hálózati konfigurációját használja.

Minden nagy adatbetöltés előtt üzenet jelenik meg. Hiba esetén ne folytassa az inferenciával. Újrafuttatáskor a korábbi osztályozót előbb felszabadítja a program. A pickle- és NumPy-objektumfájlok a telepített csomag megbízható állományai.

In [ ]:
if "classifier" in globals():
    classifier.deinit_accelerator()
    del classifier
# Avoid holding old weight dictionaries while reloading this cell.
for variable in ("conv0_weights", "fc_weights"):
    globals().pop(variable, None)
gc.collect()

print("Loading first-convolution weights...", flush=True)
conv0_weights = np.load(str(QNN_PARAM_DIR / "dorefanet-conv0.npy"),
                       encoding="latin1", allow_pickle=True).item()
print("Loading fully connected weights; please wait...", flush=True)
fc_weights = np.load(str(QNN_PARAM_DIR / "dorefanet-fc-normalized.npy"),
                    encoding="latin1", allow_pickle=True).item()
for key in ("conv0/W", "conv0/T"):
    if key not in conv0_weights:
        raise KeyError("Missing weight: " + key)
for key in ("fc0/Wn", "fc0/bn", "fc1/Wn", "fc1/bn", "fct/W"):
    if key not in fc_weights:
        raise KeyError("Missing weight: " + key)

with LABEL_FILE.open("rb") as label_stream:
    classes = pickle.load(label_stream, encoding="latin1")
if set(classes.keys()) != set(range(1000)):
    raise ValueError("Expected ImageNet labels indexed from 0 to 999.")
names = {index: classes[index][1].split(",")[0] for index in classes}
synset_to_index = {classes[index][0]: index for index in classes}

print("Initializing the FPGA...", flush=True)
classifier = Dorefanet()
try:
    net = classifier.load_network()
    classifier.init_accelerator()
except BaseException:
    classifier.deinit_accelerator()
    raise
print("Model ready. ImageNet classes:", len(classes))

## 4. Címkézett mintakép kiválasztása és megjelenítése

Hagyja az `IMAGE_NAME = None` beállítást egy véletlenszerű mintához. Konkrét minta kiválasztásához adjon meg egy létező fájlnevet az `imagenet-samples` könyvtárból. A cella megjeleníti a képet és az ismert osztályt, majd előkészíti a modell bemenetét.

Másik kép vizsgálatához futtassa újra ezt a cellát és az **5–9. szakaszt**. A 3. szakasz súlyait nem kell ismét betölteni. Az előző kép időmérései törlődnek.

In [ ]:
IMAGE_NAME = None  # Example: replace None with an existing filename string.
image_path = random.choice(image_files) if IMAGE_NAME is None else SAMPLE_DIR / IMAGE_NAME
if not image_path.is_file():
    raise FileNotFoundError("Image not found: " + str(image_path))
img, img_class = classifier.load_image(str(image_path))
expected_index = synset_to_index.get(img_class)
print("Image:", image_path.name)
print("Correct class:", names[expected_index] if expected_index is not None
      else "not available from this filename")
with Image.open(str(image_path)) as opened:
    preview = opened.convert("RGB")
preview.thumbnail((480, 360))
display(preview)
stage_seconds = {}

## 5. Az első konvolúció végrehajtása az ARM processzoron

Futtassa a következő cellát. Ez a réteg az eredeti, teljes pontosságú súlyokat használja. A kimenet küszöböléssel a kvantált FPGA-rétegek által megkövetelt reprezentációvá alakul. A kijelzett idő a konvolúciót és a küszöbölést is tartalmazza.

In [ ]:
start = time.perf_counter()
conv0 = utils.conv_layer(img, conv0_weights["conv0/W"], stride=4)
conv0 = utils.threshold(conv0, conv0_weights["conv0/T"])
stage_seconds["First convolution + threshold (CPU)"] = time.perf_counter() - start
print("CPU first stage: {:.3f} s".format(
    stage_seconds["First convolution + threshold (CPU)"]))

## 6. Kvantált rétegek az FPGA-n

Futtassa a következő cellát. Ugyanaz a gyorsító hajtja végre a kvantált rétegek konfigurált sorozatát. A pufferek előkészítése és a kimenet konverziója nem része a mért inferenciahívásnak.

A mérés az **offload-hívás** idejét jellemzi, beleértve a futtatókörnyezet többletköltségét, nem csupán az FPGA logikai műveleteinek ciklusait.

In [ ]:
out_dim = net["merge4"]["output_dim"]
out_ch = net["merge4"]["output_channels"]
conv_output = classifier.get_accel_buffer(out_ch, out_dim)
conv_input = np.ascontiguousarray(classifier.prepare_buffer(conv0))
start = time.perf_counter()
classifier.inference(conv_input, conv_output)
stage_seconds["Quantized-layer offload (FPGA)"] = time.perf_counter() - start
conv_output = classifier.postprocess_buffer(conv_output)
if not np.all(np.isfinite(conv_output)):
    raise ValueError("Non-finite accelerator output. Check model initialization.")
print("FPGA offload call: {:.3f} s".format(
    stage_seconds["Quantized-layer offload (FPGA)"]))

## 7. Teljesen összekapcsolt rétegek az ARM processzoron

Futtassa a következő cellát. Az eredeti hálózat normalizálási és teljesen összekapcsolt rétegeinek számításai változatlanok. A kizárólag nullákból álló FPGA-kimenetet külön jelzi, így nem jut hibás eredmény a nullával történő osztásig.

In [ ]:
scale = np.max(conv_output)
if scale <= 0:
    raise RuntimeError("Accelerator output is all zero. Stop and check initialization before continuing.")
fc_input = conv_output / scale
if fc_input.size != fc_weights["fc0/Wn"].shape[0]:
    raise ValueError("Accelerator output size does not match the fully connected weights.")
start = time.perf_counter()
fc0_out = utils.fully_connected(fc_input, fc_weights["fc0/Wn"], fc_weights["fc0/bn"])
fc0_out = utils.quantize(utils.qrelu(fc0_out), 2)
fc1_out = utils.fully_connected(fc0_out, fc_weights["fc1/Wn"], fc_weights["fc1/bn"])
fc1_out = utils.qrelu(fc1_out)
fct_b = np.zeros((fc_weights["fct/W"].shape[1],))
fct_out = utils.fully_connected(fc1_out, fc_weights["fct/W"], fct_b)
stage_seconds["Fully connected layers (CPU)"] = time.perf_counter() - start
print("CPU fully connected stages: {:.3f} s".format(
    stage_seconds["Fully connected layers (CPU)"]))

## 8. Az osztályozás eredményének értelmezése

Futtassa a következő cellát. Megjelenik az **öt legnagyobb pontszámú osztály**, a hozzájuk tartozó softmax-érték, valamint a Top-1 és Top-5 eredmény összevetése a kép ismert címkéjével.

A pontszámok a modell kimenetei, nem garantálják a helyes osztályozást. Egyetlen képen elért siker nem jelenti a teljes adathalmazra érvényes pontosságot.

In [ ]:
if np.asarray(fct_out).shape != (len(classes),) or not np.all(np.isfinite(fct_out)):
    raise ValueError("Expected 1,000 finite classification scores.")
out = utils.softmax(fct_out)
topn = utils.get_topn_indexes(out, 5)
print("Correct class:", names[expected_index] if expected_index is not None else "unknown")
for index in topn:
    print("{:25s} {:7.2%}".format(names[int(index)], float(out[index])))
if expected_index is not None:
    print("Top-1:", "correct" if int(topn[0]) == expected_index else "wrong")
    print("Top-5:", "correct class included" if expected_index in topn
          else "correct class not included")
fig, ax = plt.subplots(figsize=(8, 3.5))
positions = np.arange(len(topn))
colors = ["green" if int(index) == expected_index else "steelblue" for index in topn]
ax.barh(positions, out[topn], color=colors)
ax.set_yticks(positions)
ax.set_yticklabels([names[int(index)] for index in topn])
ax.invert_yaxis()
ax.set_xlim(0, min(1.0, float(np.max(out[topn])) * 1.25))
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("Softmax score (axis scaled to the displayed predictions)")
ax.set_title("Top-5 predictions (green = known correct class)")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
plt.show()

## 9. A mért futási idők összehasonlítása

Futtassa a következő cellát. A három mért számítási szakasz és az idők összege jelenik meg. A CPU és az FPGA **eltérő rétegeket** hajt végre: ez feladatmegosztás, nem ugyanazon művelet CPU–FPGA gyorsulásának mérése.

Az összeg nem tartalmazza a kép betöltését, a pufferek előkészítését és konverzióját, a softmaxot, a grafikonok készítését és a notebookcellák közötti várakozást. Nem tekinthető kamera-FPS-nek vagy az egész alkalmazás áteresztőképességének. A mérési adatokat nem osztjuk meg külön fájlban más notebookokkal.

In [ ]:
stage_order = ["First convolution + threshold (CPU)",
               "Quantized-layer offload (FPGA)",
               "Fully connected layers (CPU)"]
if any(name not in stage_seconds for name in stage_order):
    raise RuntimeError("Run all three computation stages for the current image first.")
values = [stage_seconds[name] for name in stage_order]
for name, seconds in zip(stage_order, values):
    print("{}: {:.3f} s".format(name, seconds))
print("Sum of measured stages: {:.3f} s".format(sum(values)))
fig, ax = plt.subplots(figsize=(9, 3.5))
positions = np.arange(len(stage_order))
ax.barh(positions, values, color=["steelblue", "darkorange", "steelblue"])
ax.set_yticks(positions)
ax.set_yticklabels(stage_order)
ax.invert_yaxis()
ax.set_xlabel("Measured stage time (seconds)")
ax.set_title("CPU and FPGA stages of the same inference")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
plt.show()

## 10. Befejezés és erőforrások felszabadítása

A tesztelés után futtassa a következő cellát. A modell saját erőforrás-felszabadító metódusa fut le, a nagy köztes tömbök és súlyszótárak eltávolításával. Nincs `Xlnk` import és nincs globális eszköz-reset.

A felszabadítás utáni új futtatáshoz kezdje a 3. szakasznál. Másik FPGA-notebook indítása előtt állítsa le ezt a kernelt.

In [ ]:
if "classifier" in globals():
    classifier.deinit_accelerator()
    del classifier
for variable in ("conv0_weights", "fc_weights", "img", "conv0", "conv_input",
                 "conv_output", "fc_input", "fc0_out", "fc1_out", "fct_out"):
    globals().pop(variable, None)
gc.collect()
print("Classifier released and large Python arrays cleared.")

**Validálás:** a fájlútvonalak kezelése, a súlyok és címkék betöltése, a Python-feldolgozási folyamat, a grafikonok, az ismételt inicializálás és az erőforrások felszabadítása off-board, szimulált gyorsítóval lett ellenőrizve. A numerikus hálózati műveletek az eredeti példából származnak. A natív könyvtárak, a valódi FPGA-futtatás és az ImageNet-predikciók további PYNQ-hardveres validálást igényelnek.